In [23]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

load_dotenv(".env")

try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("LogisticsEDA")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory",          "2g")
    .config("spark.driver.host",            "127.0.0.1")
    .config("spark.driver.bindAddress",     "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

DATA_PROCESSED = "file://" + os.path.abspath(os.path.join("..", "data", "processed"))

df_orders = spark.read.parquet(DATA_PROCESSED + "/orders_clean")
df_events = spark.read.parquet(DATA_PROCESSED + "/events_clean")

print(f"Config loaded from .env")
print(f"orders_clean : {df_orders.count():,} rows")
print(f"events_clean : {df_events.count():,} rows")

Config loaded from .env
orders_clean : 45,593 rows
events_clean : 42,247 rows


In [24]:
# Descriptive Statistics

print("Descriptive Statistics")
df_orders.select(
    "time_taken_min", "distance_km", "speed_kmpm", "rating", "age"
).describe().show(truncate=False)

Descriptive Statistics
+-------+-----------------+------------------+-------------------+-------------------+------------------+
|summary|time_taken_min   |distance_km       |speed_kmpm         |rating             |age               |
+-------+-----------------+------------------+-------------------+-------------------+------------------+
|count  |45593            |45593             |45593              |43438              |43463             |
|mean   |26.29460662821047|99.30391081988572 |4.367422230241152  |4.636364           |29.588983733290384|
|stddev |9.383806145163113|1099.7312805885792|52.46239642189998  |0.29645629309693183|5.8235240660821255|
|min    |10.0             |1.4650674052308583|0.02826684129430845|2.60               |20                |
|max    |54.0             |19692.674606473032|1790.0651775043884 |5.00               |39                |
+-------+-----------------+------------------+-------------------+-------------------+------------------+



In [25]:
# Null Audit

print("Null and Missing Value Audit")
total = df_orders.count()

null_rows = []
for c in df_orders.columns:
    null_count = df_orders.filter(F.col(c).isNull()).count()
    if null_count > 0:
        null_rows.append((c, null_count, round(null_count / total * 100, 2)))

if null_rows:
    print(f"  {'Column':<25} {'Nulls':>8}  {'%':>6}")
    print(f"  {'-'*42}")
    for col, n, pct in sorted(null_rows, key=lambda x: -x[1]):
        print(f"  {col:<25} {n:>8}  {pct:>5}%")
else:
    print("  No nulls found!")

Null and Missing Value Audit
  Column                       Nulls       %
  ------------------------------------------
  rating                        2155   4.73%
  age                           2130   4.67%
  multiple_deliveries            993   2.18%


In [26]:
# Delivery Time by Weather Condition

print("Avg Delivery Time by Weather Condition")
(
    df_orders.groupBy("weather_conditions")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("time_taken_min"), 2).alias("avg_time_min"),
        F.round(F.min("time_taken_min"), 2).alias("min_time_min"),
        F.round(F.max("time_taken_min"), 2).alias("max_time_min"),
    )
    .orderBy("avg_time_min", ascending=False)
    .show(truncate=False)
)

Avg Delivery Time by Weather Condition
+------------------+------------+------------+------------+------------+
|weather_conditions|total_orders|avg_time_min|min_time_min|max_time_min|
+------------------+------------+------------+------------+------------+
|fog               |7654        |28.92       |10.0        |54.0        |
|cloudy            |7536        |28.92       |10.0        |54.0        |
|nan               |616         |26.55       |10.0        |54.0        |
|windy             |7422        |26.12       |10.0        |54.0        |
|sandstorms        |7495        |25.88       |10.0        |54.0        |
|stormy            |7586        |25.87       |10.0        |54.0        |
|sunny             |7284        |21.86       |10.0        |54.0        |
+------------------+------------+------------+------------+------------+



In [27]:
# Delivery Time by Traffic Density

print("Avg Delivery Time by Traffic Density")
(
    df_orders.groupBy("road_traffic_density")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("time_taken_min"), 2).alias("avg_time_min"),
        F.round(F.min("time_taken_min"), 2).alias("min_time_min"),
        F.round(F.max("time_taken_min"), 2).alias("max_time_min"),
    )
    .orderBy("avg_time_min", ascending=False)
    .show(truncate=False)
)

Avg Delivery Time by Traffic Density
+--------------------+------------+------------+------------+------------+
|road_traffic_density|total_orders|avg_time_min|min_time_min|max_time_min|
+--------------------+------------+------------+------------+------------+
|jam                 |14143       |31.18       |10.0        |54.0        |
|high                |4425        |27.24       |10.0        |54.0        |
|medium              |10947       |26.7        |10.0        |49.0        |
|nan                 |601         |26.54       |10.0        |54.0        |
|low                 |15477       |21.27       |10.0        |44.0        |
+--------------------+------------+------------+------------+------------+



In [28]:
# Delivery Time by City Type

print("Avg Delivery Time by City Type")
(
    df_orders.groupBy("city")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("time_taken_min"), 2).alias("avg_time_min"),
        F.round(F.avg("distance_km"), 2).alias("avg_dist_km"),
        F.round(F.avg("speed_kmpm"), 4).alias("avg_speed_kmpm"),
    )
    .orderBy("avg_time_min", ascending=False)
    .show(truncate=False)
)

Avg Delivery Time by City Type
+-------------+------------+------------+-----------+--------------+
|city         |total_orders|avg_time_min|avg_dist_km|avg_speed_kmpm|
+-------------+------------+------------+-----------+--------------+
|semi-urban   |164         |49.73       |62.51      |1.2613        |
|metropolitian|34093       |27.32       |94.72      |3.925         |
|urban        |10136       |22.98       |106.5      |5.3904        |
|nan          |1200        |22.06       |173.81     |8.7199        |
+-------------+------------+------------+-----------+--------------+



In [29]:
# Driver Performance Ranking

print("Top 10 Fastest Drivers (min 5 trips)")
(
    df_orders.groupBy("driver_id", "rating")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("time_taken_min"), 2).alias("avg_time_min"),
        F.round(F.avg("distance_km"), 2).alias("avg_dist_km"),
    )
    .filter(F.col("total_trips") >= 5)
    .orderBy("avg_time_min")
    .limit(10)
    .show(truncate=False)
)

print("\nDriver Speed Rank (Window Function)")
w = Window.orderBy(F.col("avg_speed").desc())
(
    df_orders.groupBy("driver_id")
    .agg(F.round(F.avg("speed_kmpm"), 4).alias("avg_speed"))
    .filter(F.col("avg_speed").isNotNull())
    .withColumn("speed_rank", F.rank().over(w))
    .filter(F.col("speed_rank") <= 10)
    .show(truncate=False)
)

Top 10 Fastest Drivers (min 5 trips)
+--------------+------+-----------+------------+-----------+
|driver_id     |rating|total_trips|avg_time_min|avg_dist_km|
+--------------+------+-----------+------------+-----------+
|KOLRES06DEL03 |4.80  |9          |19.44       |9.08       |
|DEHRES20DEL03 |4.60  |10         |19.6        |8.49       |
|DEHRES17DEL01 |4.20  |12         |19.75       |7.96       |
|KOLRES01DEL03 |4.80  |11         |19.82       |8.67       |
|KNPRES01DEL01 |4.20  |11         |20.0        |8.27       |
|BHPRES15DEL03 |4.80  |7          |20.14       |7.77       |
|LUDHRES18DEL02|3.90  |10         |20.2        |8.06       |
|KOLRES14DEL02 |4.60  |13         |20.62       |395.32     |
|GOARES05DEL03 |5.00  |8          |20.75       |8.26       |
|KOCRES16DEL03 |4.80  |6          |21.0        |9.36       |
+--------------+------+-----------+------------+-----------+


Driver Speed Rank (Window Function)
+--------------+---------+----------+
|driver_id     |avg_speed|speed_r

26/04/28 09:31:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/28 09:31:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/28 09:31:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/28 09:31:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/28 09:31:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/28 09:31:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/28 0

In [30]:
# Peak Hour Analysis

print("Order Volume & Avg Time by Hour of Day")
(
    df_orders.filter(F.col("hour_of_day").isNotNull())
    .groupBy("hour_of_day")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("time_taken_min"), 2).alias("avg_time_min"),
    )
    .orderBy("hour_of_day")
    .show(24, truncate=False)
)

Order Volume & Avg Time by Hour of Day
+-----------+------------+------------+
|hour_of_day|total_orders|avg_time_min|
+-----------+------------+------------+
|0          |2161        |25.49       |
|8          |1818        |19.58       |
|9          |1947        |19.54       |
|10         |1991        |19.48       |
|11         |1962        |26.38       |
|12         |892         |26.83       |
|13         |784         |27.54       |
|14         |791         |27.54       |
|15         |873         |23.19       |
|16         |709         |22.94       |
|17         |4278        |27.41       |
|18         |4480        |27.27       |
|19         |4595        |30.82       |
|20         |4539        |31.19       |
|21         |4686        |31.09       |
|22         |4576        |23.2        |
|23         |4511        |22.45       |
+-----------+------------+------------+



In [31]:
# Delay Bucket Distribution

print("Delay Bucket Distribution")
total = df_orders.count()
(
    df_orders.groupBy("delay_bucket")
    .agg(F.count("*").alias("total_orders"))
    .withColumn("percentage", F.round(F.col("total_orders") / total * 100, 1))
    .orderBy("total_orders", ascending=False)
    .show(truncate=False)
)

Delay Bucket Distribution
+------------+------------+----------+
|delay_bucket|total_orders|percentage|
+------------+------------+----------+
|normal      |28492       |62.5      |
|fast        |12509       |27.4      |
|slow        |4592        |10.1      |
+------------+------------+----------+



In [32]:
# Vehicle Type Comparison

print("Vehicle Type vs Avg Delivery Time")
(
    df_orders.groupBy("type_of_vehicle")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("time_taken_min"), 2).alias("avg_time_min"),
        F.round(F.avg("distance_km"), 2).alias("avg_dist_km"),
        F.round(F.avg("speed_kmpm"), 4).alias("avg_speed_kmpm"),
    )
    .orderBy("avg_time_min")
    .show(truncate=False)
)

Vehicle Type vs Avg Delivery Time
+----------------+------------+------------+-----------+--------------+
|type_of_vehicle |total_orders|avg_time_min|avg_dist_km|avg_speed_kmpm|
+----------------+------------+------------+-----------+--------------+
|electric_scooter|3814        |24.47       |194.08     |8.614         |
|scooter         |15276       |24.48       |110.09     |5.042         |
|bicycle         |68          |26.43       |4182.0     |162.5854      |
|motorcycle      |26435       |27.61       |68.89      |2.9579        |
+----------------+------------+------------+-----------+--------------+



In [33]:
# Event Funnel (MongoDB data)

print("Delivery Event Funnel")
total_orders = df_events.select("order_id").distinct().count()
print(f"Total unique orders in events: {total_orders:,}\n")

(
    df_events.groupBy("event_type")
    .agg(F.countDistinct("order_id").alias("orders_reached"))
    .withColumn("funnel_pct",
        F.round(F.col("orders_reached") / total_orders * 100, 1))
    .orderBy("orders_reached", ascending=False)
    .show(truncate=False)
)

Delivery Event Funnel
Total unique orders in events: 11,399

+------------+--------------+----------+
|event_type  |orders_reached|funnel_pct|
+------------+--------------+----------+
|order_placed|11399         |100.0     |
|picked_up   |11399         |100.0     |
|delivered   |10269         |90.1      |
|in_transit  |8050          |70.6      |
|failed      |1130          |9.9       |
+------------+--------------+----------+



In [34]:
# Festival Impact

print("Festival vs Non-Festival Delivery Time")
(
    df_orders.groupBy("festival")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.avg("time_taken_min"), 2).alias("avg_time_min"),
        F.round(F.avg("distance_km"), 2).alias("avg_dist_km"),
        F.round(F.avg("speed_kmpm"), 4).alias("avg_speed_kmpm"),
    )
    .orderBy("festival")
    .show(truncate=False)
)

Festival vs Non-Festival Delivery Time
+--------+------------+------------+-----------+--------------+
|festival|total_orders|avg_time_min|avg_dist_km|avg_speed_kmpm|
+--------+------------+------------+-----------+--------------+
|NaN     |228         |11.17       |7.38       |0.6679        |
|No      |44469       |25.98       |98.44      |4.4029        |
|Yes     |896         |45.52       |165.45     |3.5468        |
+--------+------------+------------+-----------+--------------+

